In [59]:
import sys, os, math, torch, time
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.utils as U
import torch; 
from typing import Tuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

In [38]:
print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)
print(sys.executable); 
print(os.environ.get('CONDA_DEFAULT_ENV'))

2.2.2 True 12.1
d:\python-envs\deepLob\python.exe
d:\python-envs\deepLob


In [39]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [40]:
# загрузка данных
# путь к файлу
folder = "data"
trainFileName = "Si-12.25_2025-09-22_features.npy"
labelFileName = "Si-12.25_2025-09-22_prices.csv"

# загрузка фичей
X = np.load(os.path.join(folder, trainFileName))

print("Форма массива:", X.shape)
print("Тип данных:", X.dtype)

# загрузка меток
df_p = pd.read_csv(os.path.join(folder, labelFileName))
assert {"ms","mid"}.issubset(df_p.columns), "Нужны колонки ms и mid в prices.csv"
ms  = df_p["ms"].values.astype(np.int64)
mid = df_p["mid"].values.astype(np.float64)

assert len(X) == len(ms) == len(mid), f"Несовпадение длин: X={len(X)} ms={len(ms)} mid={len(mid)}"

Форма массива: (143799, 138)
Тип данных: float32


In [41]:
# ==== функции ====
def build_barrier_labels(ms: np.ndarray,
                         mid: np.ndarray,
                         tick_size: float = 1.0,
                         theta_ticks: int = 1,
                         horizon_sec: float = 1.5) -> np.ndarray:
    """
    Triple-barrier по реальному времени t -> t+τ:
      класс 2: цена поднималась >= +θ,
      класс 0: цена падала  <= -θ,
      класс 1: ни то ни другое (flat).
    Там, где нет будущего окна, возвращаем -1.
    """
    N = len(mid)
    y = np.full(N, -1, dtype=np.int64)
    theta = theta_ticks * tick_size

    import bisect
    for i in range(N):
        t0 = ms[i]
        t_end = t0 + int(horizon_sec * 1000)
        j = bisect.bisect_right(ms, t_end, lo=i+1)
        if j <= i+1:
            continue
        m0 = mid[i]
        w = mid[i+1:j]
        if w.size == 0:
            continue
        up_hit = (w.max() - m0) >= theta
        dn_hit = (w.min() - m0) <= -theta
        if up_hit and not dn_hit:
            y[i] = 2
        elif dn_hit and not up_hit:
            y[i] = 0
        elif up_hit and dn_hit:
            # кто наступил раньше (грубо)
            up_idx = np.argmax(w == w.max())
            dn_idx = np.argmax(w == w.min())
            y[i] = 2 if up_idx < dn_idx else 0
        else:
            y[i] = 1
    return y

def make_windows(X2D: np.ndarray, T: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Скользящие окна по времени (каузально):
      вход: X2D (N, F)
      выход: Xwin (N-T+1, T, F), end_idx — индексы последних точек окон
    """
    N, F = X2D.shape
    if N < T:
        raise ValueError(f"Мало данных для окна: N={N} < T={T}")
    s0, s1 = X2D.strides
    Xwin = np.lib.stride_tricks.as_strided(
        X2D, shape=(N - T + 1, T, F), strides=(s0, s0, s1)
    ).copy()
    end_idx = np.arange(T-1, N)
    return Xwin, end_idx

In [42]:
# ==== параметры задачи ====
TICK_SIZE   = 1.0   # тик инструмента
THETA_TICKS = 5     # порог в тиках (±1)
HORIZON_SEC = 2     # горизонт (сек)
T           = 240    # длина окна (шагов), умнодить на время между срезами (0.3сек), полученное значение должно быть в 10-40 раз больше HORIZON_SEC

In [43]:
# ==== построение меток и окон ====
y_all = build_barrier_labels(ms, mid, tick_size=TICK_SIZE,
                             theta_ticks=THETA_TICKS, horizon_sec=HORIZON_SEC)

valid = (y_all != -1)
Xv, yv, msv, midv = X[valid], y_all[valid], ms[valid], mid[valid]

Xwin, end_idx = make_windows(Xv, T)
y_win  = yv[end_idx]
ms_win = msv[end_idx]

print(f"Классы и счётчики: {np.unique(y_win, return_counts=True)}")
print("Окна:", Xwin.shape, "| метки:", y_win.shape)

Классы и счётчики: (array([0, 1, 2], dtype=int64), array([  6596, 128575,   6694], dtype=int64))
Окна: (141865, 240, 138) | метки: (141865,)


In [44]:
# ==== временной сплит: train / val / test ====
Nw = len(Xwin)
i_tr = int(Nw * 0.70)
i_va = int(Nw * 0.85)

X_train, y_train = Xwin[:i_tr], y_win[:i_tr]
X_val,   y_val   = Xwin[i_tr:i_va], y_win[i_tr:i_va]
X_test,  y_test  = Xwin[i_va:],     y_win[i_va:]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (99305, 240, 138) Val: (21280, 240, 138) Test: (21280, 240, 138)


In [61]:
# 1) Dataset-обёртка: из numpy -> тензоры нужной формы для Conv1d
# Conv1d ждёт вход (batch, C, T), где C = кол-во признаков (F), T = длина окна
class NpWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32, copy=False)
        self.y = y.astype(np.int64,  copy=False)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        # Conv1d ждёт (B, C, T) → тут делаем (F, T)
        x = torch.from_numpy(self.X[i].T.copy())  # (F, T)
        y = torch.tensor(self.y[i])
        return x, y

# Новые подпрыжки
# классовые веса и sampler для train
cls, cnt = np.unique(y_train, return_counts=True)
print("Train class counts:", dict(zip(cls, cnt)))
class_weights_inv = cnt.sum() / (len(cls) * cnt)   # простая инверсия
class_weights = np.ones(3, dtype=np.float32)
for c, w in zip(cls, class_weights_inv):
    class_weights[int(c)] = float(w)

# веса на уровне примеров (для sampler)
sample_weights = class_weights[y_train]
sampler = WeightedRandomSampler(weights=sample_weights,
                                num_samples=len(sample_weights),
                                replacement=True)

# 2) DataLoader'ы
batch_size = 512
#train_dl = DataLoader(NpWindowDataset(X_train, y_train), batch_size=batch_size, sampler=sampler, drop_last=True)

# --- БЕЗ WeightedRandomSampler ---
train_dl = DataLoader(NpWindowDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)
val_dl   = DataLoader(NpWindowDataset(X_val,   y_val),   batch_size=batch_size, shuffle=False)
test_dl  = DataLoader(NpWindowDataset(X_test,  y_test),  batch_size=batch_size, shuffle=False)


# 3) DeepLOB-like модель на Conv1d (по времени)
class TemporalConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=5, d=1, groups=8):
        super().__init__()
        pad = (k - 1) * d
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=k, dilation=d, padding=pad)
        self.act1  = nn.GELU()
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, dilation=2, padding=2*2)
        self.act2  = nn.GELU()
        # GroupNorm не зависит от размера батча и «состава» батча
        g = min(groups, out_ch)
        self.norm  = nn.GroupNorm(num_groups=g, num_channels=out_ch)

    def forward(self, x):
        y = self.conv1(x); y = self.act1(y)
        y = self.conv2(y); y = self.act2(y)
        y = self.norm(y)
        return y[..., :x.shape[-1]]  # каузально

class DeepLOBLike(nn.Module):
    def __init__(self, F, num_classes=3, hidden=128, groups=8, p_drop=0.3):
        super().__init__()
        self.stem = nn.Conv1d(F, hidden, kernel_size=1)
        self.b1 = TemporalConvBlock(hidden, hidden, k=5, d=1, groups=groups)
        self.b2 = TemporalConvBlock(hidden, hidden, k=5, d=2, groups=groups)
        self.b3 = TemporalConvBlock(hidden, hidden, k=5, d=4, groups=groups)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.b1(x); x = self.b2(x); x = self.b3(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)
        
# =========================
# Лоссы: FocalLoss (по умолчанию) или CrossEntropy
# =========================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = None if alpha is None else torch.tensor(alpha, dtype=torch.float32)
        self.gamma = gamma
        self.reduction = reduction
    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(
            logits, targets, reduction='none',
            weight=self.alpha.to(logits.device) if self.alpha is not None else None
        )
        pt = torch.exp(-ce)               # prob(true class)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.reduction == "mean": return loss.mean()
        if self.reduction == "sum":  return loss.sum()
        return loss

Train class counts: {0: 4948, 1: 89348, 2: 5009}


In [62]:
# --- конфиги обучения ---
use_focal = True
# усиливаем up/down, ослабляем flat (подбери позже по валидации)
alpha = [8.0, 0.15, 8.0]  # [down, flat, up]
gamma = 2.5
learning_rate = 1e-4
max_norm = 0.9 # (подбери 0.5–5.0)

# =========================
# Инициализация
# =========================
_, T, F = X_train.shape
model = DeepLOBLike(F=F, num_classes=3, hidden=128).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=3e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

# Focal с усилением up/down и ослаблением flat
#criterion = FocalLoss(alpha=alpha, gamma=gamma)
# не занижаем flat (наоборот, чуть поддержим), чтобы модель СНОВА ЕГО ПРЕДСКАЗЫВАЛА
ce_weights = torch.tensor([1.0, 0.7, 1.0], dtype=torch.float32, device=device)  # [down, flat, up]
criterion = nn.CrossEntropyLoss(weight=ce_weights)

# =========================
# Функции валидации и теста (macro-F1)
# =========================
def eval_loader(dl):
    model.eval()
    total, n = 0.0, 0
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = nn.functional.cross_entropy(logits, yb, reduction='mean')  # для мониторинга
            total += loss.item() * xb.size(0); n += xb.size(0)
            y_true.append(yb.cpu().numpy())
            y_pred.append(logits.argmax(1).cpu().numpy())
    y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred)
    val_loss = total / max(1, n)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    f1_down = f1_score(y_true, y_pred, labels=[0], average=None)[0]
    f1_flat = f1_score(y_true, y_pred, labels=[1], average=None)[0]
    f1_up   = f1_score(y_true, y_pred, labels=[2], average=None)[0]
    return val_loss, macro_f1, (f1_down, f1_flat, f1_up)

In [63]:
# =========================
# Тренировка + Early Stopping по macro-F1
# =========================
def evaluate(dl):
    model.eval()
    total, n = 0.0, 0
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            # для мониторинга используем обычный CE
            loss = nn.functional.cross_entropy(logits, yb, reduction='mean')
            total += loss.item() * xb.size(0); n += xb.size(0)
            y_true.append(yb.cpu().numpy())
            y_pred.append(logits.argmax(1).cpu().numpy())
    y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred)
    val_loss = total / max(1, n)
    macroF1 = f1_score(y_true, y_pred, average="macro")
    f1d = f1_score(y_true, y_pred, labels=[0], average=None)[0]
    f1f = f1_score(y_true, y_pred, labels=[1], average=None)[0]
    f1u = f1_score(y_true, y_pred, labels=[2], average=None)[0]
    cm  = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    return val_loss, macroF1, (f1d, f1f, f1u), cm

best_score, best_state = -1.0, None
epochs = 20
for ep in range(1, epochs+1):
    model.train()
    total, correct, n = 0.0, 0, 0
    tic = time.time()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        # клип градиента — помогает от «скачков»
        scaler.unscale_(optimizer)
        U.clip_grad_norm_(model.parameters(), max_norm)
        scaler.step(optimizer); scaler.update()
        total += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        n += xb.size(0)

    train_loss = total / n
    train_acc  = correct / n
    scheduler.step()

    val_loss, macroF1, (f1d,f1f,f1u), cm = evaluate(val_dl)
    print(f"[{ep:02d}] train_loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} macroF1={macroF1:.3f} "
          f"F1↓={f1d:.3f} F1○={f1f:.3f} F1↑={f1u:.3f}  ({time.time()-tic:.1f}s)")
    # можешь разок вывести матрицу ошибок для понимания перекосов
    if ep in (1, 5, 10, 15, 20):
        print("Confusion matrix [rows=true, cols=pred]:\n", cm)

    if macroF1 > best_score:
        best_score = macroF1
        best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}

[01] train_loss=0.4998 acc=0.900 | val_loss=0.4939 macroF1=0.309 F1↓=0.000 F1○=0.927 F1↑=0.000  (25.8s)
Confusion matrix [rows=true, cols=pred]:
 [[    0  1433     0]
 [    0 18383     0]
 [    0  1464     0]]
[02] train_loss=0.4875 acc=0.900 | val_loss=0.4920 macroF1=0.309 F1↓=0.000 F1○=0.927 F1↑=0.000  (25.0s)
[03] train_loss=0.4824 acc=0.900 | val_loss=0.4867 macroF1=0.309 F1↓=0.000 F1○=0.927 F1↑=0.000  (25.2s)
[04] train_loss=0.4792 acc=0.900 | val_loss=0.4839 macroF1=0.309 F1↓=0.000 F1○=0.927 F1↑=0.000  (25.0s)
[05] train_loss=0.4763 acc=0.900 | val_loss=0.4870 macroF1=0.309 F1↓=0.000 F1○=0.927 F1↑=0.000  (25.0s)
Confusion matrix [rows=true, cols=pred]:
 [[    0  1433     0]
 [    0 18383     0]
 [    0  1464     0]]
[06] train_loss=0.4698 acc=0.900 | val_loss=0.4817 macroF1=0.309 F1↓=0.000 F1○=0.927 F1↑=0.000  (25.2s)
[07] train_loss=0.4636 acc=0.900 | val_loss=0.4810 macroF1=0.309 F1↓=0.000 F1○=0.927 F1↑=0.000  (25.2s)
[08] train_loss=0.4611 acc=0.900 | val_loss=0.4976 macroF1=0

KeyboardInterrupt: 

In [30]:
# 6) Тест
if best_state is not None:
    model.load_state_dict(best_state)
model.eval()

# отчёт по тесту
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        logits = model(xb)
        pred = logits.argmax(1).cpu().numpy()
        all_preds.append(pred)
        all_true.append(yb.numpy())
all_preds = np.concatenate(all_preds)
all_true  = np.concatenate(all_true)

from sklearn.metrics import classification_report
print("\nBest val acc:", round(best_val_acc, 4))
print(classification_report(all_true, all_preds, digits=3, target_names=["down","flat","up"]))


Best val acc: 0.863
              precision    recall  f1-score   support

        down      0.002     0.009     0.003       215
        flat      0.983     0.790     0.876     20844
          up      0.018     0.267     0.034       221

    accuracy                          0.777     21280
   macro avg      0.334     0.355     0.304     21280
weighted avg      0.963     0.777     0.858     21280



In [31]:
np.unique(y_train, return_counts=True)

(array([0, 1, 2], dtype=int64), array([ 4948, 89348,  5009], dtype=int64))